# Lab 06: Support Vector Machines

Complete all nine questions in order. Put code in the code response cells and explanations in the Markdown response cells beside each prompt. Restart the kernel and run the whole notebook before submitting. Label plots with titles, axes, and legends where relevant.

This lab uses three datasets: a synthetic two-feature classification problem, two interlocking moons, and the built-in wine recognition dataset. The last part uses a synthetic imbalanced dataset. No download is required.


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_wine, make_classification, make_moons
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sns.set_theme(style="whitegrid")


## Plotting helpers

The following functions plot two-feature data, decision boundaries, support vectors, and signed decision-function contours. They are supplied so you can focus on fitting and interpreting the models. Use `ax=` to draw two models side by side. The mesh resolution is bounded to keep notebook execution practical.


In [ ]:
def plot_decision_boundary(X, y, model, *, title="Decision boundary", ax=None,
                           support_vectors=None, plot_margin=False):
    X = np.asarray(X)
    y = np.asarray(y)
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError("Decision-boundary plots require exactly two features.")

    created_figure = ax is None
    if created_figure:
        _, ax = plt.subplots(figsize=(8, 6))

    x_values = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 180)
    y_values = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 180)
    xx, yy = np.meshgrid(x_values, y_values)
    mesh = np.c_[xx.ravel(), yy.ravel()]
    predictions = model.predict(mesh).reshape(xx.shape)

    ax.contourf(xx, yy, predictions, alpha=0.25, cmap="RdYlBu")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdYlBu", edgecolors="black", s=35)

    if plot_margin:
        decisions = model.decision_function(mesh).reshape(xx.shape)
        ax.contour(xx, yy, decisions, levels=[-1, 0, 1],
                   colors="black", linestyles=["dashed", "solid", "dashed"])

    if support_vectors is not None:
        ax.scatter(support_vectors[:, 0], support_vectors[:, 1], s=130,
                   facecolors="none", edgecolors="green", linewidths=1.5,
                   label="Support vectors")
        ax.legend()

    ax.set(xlabel="Feature 1", ylabel="Feature 2", title=title)
    if created_figure:
        plt.tight_layout()
        plt.show()
    return ax


def compare_svm_parameters(X, y, parameter, values, **fixed_parameters):
    fig, axes = plt.subplots(1, len(values), figsize=(6 * len(values), 5))
    axes = np.atleast_1d(axes)
    for ax, value in zip(axes, values):
        model = SVC(**(fixed_parameters | {parameter: value})).fit(X, y)
        plot_decision_boundary(
            X, y, model, ax=ax, title=f"{parameter} = {value}",
            support_vectors=model.support_vectors_, plot_margin=True,
        )
    plt.tight_layout()
    plt.show()
    return fig, axes


## Part 1: Linear SVMs

A linear support vector classifier uses a separating hyperplane. Support vectors determine the fitted boundary; (C) controls the penalty for margin violations. Smaller (C) allows more violations and generally favors a wider margin.

### Question 1: Synthetic data and a linear boundary


In [ ]:
X, y = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.1, random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42,
)


**1(a).** Plot the training observations with different colors for the two classes. Does the sample appear linearly separable?


In [ ]:
# 1(a)
# Your code here


**Your response here:**


**1(b).** Fit `SVC(kernel='linear', C=1.0, random_state=42)` on the training set. Report training and test accuracy.


In [ ]:
# 1(b)
# Your code here


**1(c).** Report the total number of support vectors. Use `n_support_` (which gives one count per class).


In [ ]:
# 1(c)
# Your code here


**1(d).** Plot the fitted decision boundary, support vectors, and margin. You may use `plot_decision_boundary` with `support_vectors_` and `plot_margin=True`.


In [ ]:
# 1(d)
# Your code here


### Question 2: Effect of C


**2(a).** Fit three linear SVMs with `C=0.01`, `C=1.0`, and `C=100.0`, using `random_state=42` for each. Keep the fitted models for later steps.


In [ ]:
# 2(a)
# Your code here


**2(b).** For each model, report its total number of support vectors, training accuracy, and test accuracy in a readable table.


In [ ]:
# 2(b)
# Your code here


**2(c).** Show the three decision boundaries in one figure. You may use `compare_svm_parameters`. Describe how the boundaries and margins change as C increases.


In [ ]:
# 2(c)
# Your code here


**Your response here:**


**2(d).** Which C gives the highest observed test accuracy? Which model uses the fewest support vectors? Report both, including ties if present.


**Your response here:**


**2(e).** Explain how increasing C can change the bias–variance trade-off. Use your results as evidence without treating a support-vector count alone as a measure of boundary complexity.


**Your response here:**


## Part 2: Nonlinear SVMs and the kernel trick

The RBF kernel can form curved decision boundaries. Its `gamma` parameter controls how locally each training point influences the fitted boundary. Small gamma generally produces smoother boundaries; large gamma can make them more intricate.

### Question 3: Two moons


In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42,
)


**3(a).** Scatter-plot the moons training data. Could a straight line separate the classes well?


In [ ]:
# 3(a)
# Your code here


**Your response here:**


**3(b).** Fit a linear SVM (`kernel='linear'`, `C=1.0`, `random_state=42`) and report test accuracy.


In [ ]:
# 3(b)
# Your code here


**3(c).** Fit an RBF SVM (`kernel='rbf'`, `C=1.0`, `gamma='scale'`, `random_state=42`) and report test accuracy.


In [ ]:
# 3(c)
# Your code here


**3(d).** Show the linear and RBF decision boundaries side by side, using the same training data and clearly labeled panels. The plotting helper accepts an `ax` argument.


In [ ]:
# 3(d)
# Your code here


**3(e).** Explain the difference in observed performance using the shapes of the two boundaries and the moons. If RBF did not score higher, discuss that result rather than assuming it did.


**Your response here:**


### Question 4: Tune C and gamma for the RBF model


**4(a).** Use `GridSearchCV` with five folds, training data only, and accuracy scoring to search the C and gamma grid below for an RBF SVM. Keep the fitted search object.


In [ ]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, 0.1, 1, 10],
}
# Your code here


**4(b).** Report the selected C and gamma values from the fitted search.


In [ ]:
# 4(b)
# Your code here


**4(c).** Report the best mean cross-validation accuracy.


In [ ]:
# 4(c)
# Your code here


**4(d).** Use the refitted `best_estimator_` to report test accuracy. `GridSearchCV` refits the best candidate on the full training set by default; do not tune on the test set.


In [ ]:
# 4(d)
# Your code here


**4(e).** Plot a heatmap of mean cross-validation accuracy for every C–gamma combination. Describe the patterns you see.


In [ ]:
# 4(e)
# Your code here


**Your response here:**


**4(f).** Fit an RBF SVM with `C=100`, `gamma=10`, and `random_state=42`. Compare its decision boundary and training/test accuracy with the selected model. Does the evidence suggest overfitting?


In [ ]:
# 4(f)
# Your code here


**Your response here:**


## Part 3: Wine recognition with SVMs

The built-in `load_wine` data contains 178 samples, 13 chemical measurements, and three cultivar classes. For this lab, convert it to a binary target: original class 0 versus original classes 1 and 2. Scale features within a fitted pipeline so training statistics are never learned from the held-out test set.

### Question 5: Inspect and prepare the wine data


In [ ]:
wine = load_wine()
X_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
y_wine_binary = pd.Series((wine.target > 0).astype(int), name="other_cultivar")


**5(a).** Report the number of samples and features.


In [ ]:
# 5(a)
# Your code here


**5(b).** Report each binary class's count and percentage. Is the dataset approximately balanced? Explain your judgment.


In [ ]:
# 5(b)
# Your code here


**Your response here:**


**5(c).** Split the binary wine data into 70% training and 30% test using `random_state=42`. Keep the test set for final evaluation.


In [ ]:
# 5(c)
# Your code here


**5(d).** Define a scikit-learn `Pipeline` with `StandardScaler` followed by a linear `SVC(C=1.0, random_state=42)`. Use this pipeline pattern for the kernel comparison so scaling is fit on training data only.


In [ ]:
# 5(d)
# Your code here


### Question 6: Compare SVM kernels


**6(a).** Fit three scaling pipelines on the wine training data: linear (`C=1.0`), polynomial (`degree=3`, `C=1.0`), and RBF (`C=1.0`, `gamma='scale'`). Set `random_state=42` on each SVC. Save each fitted pipeline.


In [ ]:
# 6(a)
# Your code here


**6(b).** For each kernel, report training accuracy, test accuracy, total support vectors, and training time in seconds. Measure training time with `time.perf_counter()`. The fitted SVC is available through its pipeline's `named_steps`.


In [ ]:
# 6(b)
# Your code here


**6(c).** Which kernel has the highest observed test accuracy? State any ties.


**Your response here:**


**6(d).** Plot a confusion matrix for a model with the highest observed test accuracy. Label the binary classes.


In [ ]:
# 6(d)
# Your code here


**6(e).** Would you recommend an SVM for this task? Use the observed scores, confusion matrix, training time, and interpretability in your explanation.


**Your response here:**


### Question 7: Coefficients of a linear SVM


**7(a).** Fit a linear SVM in a scaling pipeline on the wine training set (or reuse the fitted linear pipeline from Question 6).


In [ ]:
# 7(a)
# Your code here


**7(b).** Extract its `coef_` vector and associate each coefficient with the corresponding feature name. The coefficients are for standardized features.


In [ ]:
# 7(b)
# Your code here


**7(c).** Draw a horizontal bar chart of absolute coefficient values for all features, sorted from largest to smallest.


In [ ]:
# 7(c)
# Your code here


**7(d).** Which three features have the largest absolute coefficients?


**Your response here:**


**7(e).** Explain why the linear SVM has one coefficient per feature and why the RBF SVM does not have the same directly interpretable `coef_` vector. Interpret magnitudes with the standardization in mind.


**Your response here:**


## Part 4: Imbalanced classification

Accuracy can hide weak minority-class performance. Compare precision, recall, F1, and confusion matrices for a standard and a class-weighted SVM. Treat class 1 as the minority class.

### Question 8: Baseline on imbalanced data


In [ ]:
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5,
    n_classes=2, weights=[0.95, 0.05], flip_y=0.01, random_state=42,
)
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb,
)


**8(a).** Report both training-set class counts and the majority-to-minority count ratio.


In [ ]:
# 8(a)
# Your code here


**8(b).** Fit a standard RBF SVM (`C=1.0`, `gamma='scale'`, `random_state=42`) in a `StandardScaler` pipeline, with no class weighting. Report test accuracy and precision, recall, and F1 for both classes using `classification_report`.


In [ ]:
# 8(b)
# Your code here


**8(c).** Assess minority-class performance. Explain why overall accuracy alone may be misleading here.


**Your response here:**


### Question 9: Class weights


**9(a).** Fit the same scaling and RBF SVM pipeline with `class_weight='balanced'`.


In [ ]:
# 9(a)
# Your code here


**9(b).** Report the same test metrics as in 8(b), and describe how minority-class performance changed.


In [ ]:
# 9(b)
# Your code here


**Your response here:**


**9(c).** Show confusion matrices for the standard and balanced models. What trade-off do you observe between missed minority cases and false alarms?


In [ ]:
# 9(c)
# Your code here


**Your response here:**


**9(d).** Use five-fold `GridSearchCV` on the training set to tune C and class weight simultaneously, using `scoring='f1'`. Keep scaling inside the pipeline, and use pipeline parameter names such as `svc__C`. Search the grid below.


In [ ]:
param_grid_imb = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__class_weight": [None, "balanced", {0: 1, 1: 10}, {0: 1, 1: 20}],
}
# Your code here


**9(e).** Report the selected C and class weight, and the corresponding mean cross-validation F1.


In [ ]:
# 9(e)
# Your code here


**9(f).** For the refitted best model, report test-set class-1 F1, precision, and recall. Compare them with the standard and balanced models without changing the selected model based on test results.


In [ ]:
# 9(f)
# Your code here


**Your response here:**


## Submission check

- Every code and written-response cell is complete.
- Figures have readable titles, axis labels, and legends where needed.
- The notebook runs in order after a kernel restart.
- `lab-06.ipynb` and your generated `uv.lock` are committed and pushed to your private submission repository.
